# Time Series Forecasting: Deep Learning vs Traditional Models Comparison

This comprehensive notebook compares various time series forecasting approaches:

## Deep Learning Models:
1. **MLP (Multilayer Perceptron)** - Feedforward neural network
2. **CNN (Convolutional Neural Network)** - 1D convolution for sequence patterns
3. **LSTM (Long Short-Term Memory)** - Recurrent neural network for sequences
4. **CNN-LSTM** - Hybrid model combining CNN and LSTM

## Traditional Time Series Models:
5. **ARIMA** - Autoregressive Integrated Moving Average
6. **SARIMA** - Seasonal ARIMA
7. **Prophet** - Facebook's time series forecasting tool
8. **Exponential Smoothing (Holt-Winters)** - Triple exponential smoothing
9. **XGBoost** - Gradient boosting for time series
10. **Random Forest** - Ensemble method for forecasting

## 1. Import Libraries and Setup

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, MaxPooling1D, Flatten, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping

# Traditional Time Series
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Prophet
from prophet import Prophet

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")

## 2. Load and Explore Data

In [ ]:
# Load data
train = pd.read_csv('train.csv', parse_dates=['date'])
test = pd.read_csv('test.csv', parse_dates=['date'])

print("Training Data Shape:", train.shape)
print("Test Data Shape:", test.shape)
print("\nTraining Data Info:")
print(train.info())

In [ ]:
# Display first few rows
print("Training Data Sample:")
train.head(10)

In [ ]:
# Statistical summary
print("Statistical Summary:")
train.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(train.isnull().sum())

In [ ]:
# Time period
print(f"Min date from train set: {train['date'].min().date()}")
print(f"Max date from train set: {train['date'].max().date()}")
print(f"Min date from test set: {test['date'].min().date()}")
print(f"Max date from test set: {test['date'].max().date()}")

## 3. Data Visualization

In [ ]:
# Aggregate daily sales
daily_sales = train.groupby('date', as_index=False)['sales'].sum()
store_daily_sales = train.groupby(['store', 'date'], as_index=False)['sales'].sum()
item_daily_sales = train.groupby(['item', 'date'], as_index=False)['sales'].sum()

print("Daily Sales Data Shape:", daily_sales.shape)

In [ ]:
# Plot overall daily sales
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Overall daily sales
axes[0, 0].plot(daily_sales['date'], daily_sales['sales'], color='blue', alpha=0.7)
axes[0, 0].set_title('Overall Daily Sales', fontsize=14)
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Sales')

# Monthly average sales
monthly_sales = daily_sales.copy()
monthly_sales['month'] = monthly_sales['date'].dt.to_period('M')
monthly_avg = monthly_sales.groupby('month')['sales'].mean()
axes[0, 1].bar(range(len(monthly_avg)), monthly_avg.values, color='green', alpha=0.7)
axes[0, 1].set_title('Monthly Average Sales', fontsize=14)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Average Sales')

# Sales by store
store_sales = train.groupby('store')['sales'].sum().sort_index()
axes[1, 0].bar(store_sales.index, store_sales.values, color='orange', alpha=0.7)
axes[1, 0].set_title('Total Sales by Store', fontsize=14)
axes[1, 0].set_xlabel('Store')
axes[1, 0].set_ylabel('Total Sales')

# Sales by item (top 20)
item_sales = train.groupby('item')['sales'].sum().sort_values(ascending=False).head(20)
axes[1, 1].barh(range(len(item_sales)), item_sales.values, color='red', alpha=0.7)
axes[1, 1].set_yticks(range(len(item_sales)))
axes[1, 1].set_yticklabels(item_sales.index)
axes[1, 1].set_title('Top 20 Items by Sales', fontsize=14)
axes[1, 1].set_xlabel('Total Sales')
axes[1, 1].set_ylabel('Item')

plt.tight_layout()
plt.savefig('output/data_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Seasonal decomposition
from statsmodels.tsa.seasonal import seasonal_decompose

# Use daily aggregated data for decomposition
daily_ts = daily_sales.set_index('date')['sales']

# Decompose with weekly seasonality (period=7)
decomposition = seasonal_decompose(daily_ts, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
decomposition.observed.plot(ax=axes[0], title='Observed')
decomposition.trend.plot(ax=axes[1], title='Trend')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal (Weekly)')
decomposition.resid.plot(ax=axes[3], title='Residual')

plt.tight_layout()
plt.savefig('output/seasonal_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preparation for Models

In [ ]:
# For fair comparison, we'll focus on a single store-item combination
# and aggregate total daily sales

# Use aggregated daily sales for comparison
df = daily_sales.copy()
df.columns = ['ds', 'y']  # Prophet format

print(f"Total data points: {len(df)}")
print(f"Date range: {df['ds'].min()} to {df['ds'].max()}")

In [ ]:
# Split data into train and validation
# Use last 90 days for validation (matching the original project's forecast horizon)
validation_days = 90

train_df = df[:-validation_days].copy()
val_df = df[-validation_days:].copy()

print(f"Training set: {len(train_df)} days")
print(f"Validation set: {len(val_df)} days")
print(f"Training range: {train_df['ds'].min()} to {train_df['ds'].max()}")
print(f"Validation range: {val_df['ds'].min()} to {val_df['ds'].max()}")

In [ ]:
# Scale data for deep learning models
scaler = MinMaxScaler()

# Fit on training data
train_scaled = scaler.fit_transform(train_df[['y']])
val_scaled = scaler.transform(val_df[['y']])

# Combine for sequence creation
full_scaled = np.vstack([train_scaled, val_scaled])

print("Data scaled successfully!")

## 5. Helper Functions

In [ ]:
# Function to create sequences for deep learning
def create_sequences(data, window_size):
    """Create sequences for time series forecasting"""
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

# Function to calculate metrics
def calculate_metrics(y_true, y_pred, model_name):
    """Calculate evaluation metrics"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape
    }

# Function to plot predictions
def plot_predictions(train_data, val_data, predictions, model_name, dates_train, dates_val):
    """Plot training, validation and predictions"""
    plt.figure(figsize=(14, 6))
    
    # Plot last 100 days of training
    plt.plot(dates_train[-100:], train_data[-100:], label='Training (last 100 days)', color='blue', alpha=0.7)
    
    # Plot validation
    plt.plot(dates_val, val_data, label='Actual Validation', color='green', alpha=0.7)
    
    # Plot predictions
    plt.plot(dates_val, predictions, label=f'{model_name} Predictions', color='red', linestyle='--', alpha=0.7)
    
    plt.title(f'{model_name} - Forecasting Results', fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Sales')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'output/{model_name.lower().replace("-", "_")}_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Helper functions defined!")

## 6. Deep Learning Models

### 6.1 MLP (Multilayer Perceptron)

In [ ]:
# Parameters
WINDOW_SIZE = 30  # Look back 30 days
EPOCHS = 100
BATCH_SIZE = 32

# Create sequences
X, y = create_sequences(train_scaled.flatten(), WINDOW_SIZE)
X = X.reshape(X.shape[0], X.shape[1])  # Flatten for MLP

# Split for validation during training
X_train_mlp, X_val_mlp, y_train_mlp, y_val_mlp = train_test_split(X, y, test_size=0.1, random_state=42)

print(f"MLP Training shape: {X_train_mlp.shape}")
print(f"MLP Validation shape: {X_val_mlp.shape}")

In [ ]:
# Build MLP Model
mlp_model = Sequential([
    Dense(100, activation='relu', input_shape=(WINDOW_SIZE,)),
    Dense(50, activation='relu'),
    Dense(25, activation='relu'),
    Dense(1)
])

mlp_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("MLP Model Summary:")
mlp_model.summary()

In [ ]:
# Train MLP
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

mlp_history = mlp_model.fit(
    X_train_mlp, y_train_mlp,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_mlp, y_val_mlp),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Make predictions on validation set
# Use the last portion of training data to predict validation period
last_sequence = train_scaled[-WINDOW_SIZE:].flatten()

mlp_predictions = []
current_seq = last_sequence.copy()

for _ in range(len(val_df)):
    pred = mlp_model.predict(current_seq.reshape(1, -1), verbose=0)[0][0]
    mlp_predictions.append(pred)
    current_seq = np.roll(current_seq, -1)
    current_seq[-1] = pred

# Inverse transform predictions
mlp_predictions = scaler.inverse_transform(np.array(mlp_predictions).reshape(-1, 1)).flatten()

# Calculate metrics
mlp_metrics = calculate_metrics(val_df['y'].values, mlp_predictions, 'MLP')
print(f"\nMLP Metrics:")
print(f"RMSE: {mlp_metrics['RMSE']:.4f}")
print(f"MAE: {mlp_metrics['MAE']:.4f}")
print(f"MAPE: {mlp_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot MLP predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    mlp_predictions, 
    'MLP',
    train_df['ds'].values,
    val_df['ds'].values
)

### 6.2 CNN (Convolutional Neural Network)

In [ ]:
# Create sequences for CNN
X_cnn, y_cnn = create_sequences(train_scaled, WINDOW_SIZE)

# Split for validation
X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn = train_test_split(X_cnn, y_cnn, test_size=0.1, random_state=42)

print(f"CNN Training shape: {X_train_cnn.shape}")
print(f"CNN Validation shape: {X_val_cnn.shape}")

In [ ]:
# Build CNN Model
cnn_model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(WINDOW_SIZE, 1)),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(50, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("CNN Model Summary:")
cnn_model.summary()

In [ ]:
# Train CNN
cnn_history = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_cnn, y_val_cnn),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Make CNN predictions
last_seq_cnn = train_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)

cnn_predictions = []
current_seq_cnn = last_seq_cnn.copy()

for _ in range(len(val_df)):
    pred = cnn_model.predict(current_seq_cnn, verbose=0)[0][0]
    cnn_predictions.append(pred)
    current_seq_cnn = np.roll(current_seq_cnn, -1)
    current_seq_cnn[0, -1, 0] = pred

# Inverse transform
cnn_predictions = scaler.inverse_transform(np.array(cnn_predictions).reshape(-1, 1)).flatten()

# Calculate metrics
cnn_metrics = calculate_metrics(val_df['y'].values, cnn_predictions, 'CNN')
print(f"\nCNN Metrics:")
print(f"RMSE: {cnn_metrics['RMSE']:.4f}")
print(f"MAE: {cnn_metrics['MAE']:.4f}")
print(f"MAPE: {cnn_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot CNN predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    cnn_predictions, 
    'CNN',
    train_df['ds'].values,
    val_df['ds'].values
)

### 6.3 LSTM (Long Short-Term Memory)

In [ ]:
# Create sequences for LSTM (same as CNN)
X_lstm, y_lstm = create_sequences(train_scaled, WINDOW_SIZE)

# Split for validation
X_train_lstm, X_val_lstm, y_train_lstm, y_val_lstm = train_test_split(X_lstm, y_lstm, test_size=0.1, random_state=42)

print(f"LSTM Training shape: {X_train_lstm.shape}")

In [ ]:
# Build LSTM Model
lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(WINDOW_SIZE, 1), return_sequences=True),
    LSTM(25, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("LSTM Model Summary:")
lstm_model.summary()

In [ ]:
# Train LSTM
lstm_history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_lstm, y_val_lstm),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Make LSTM predictions
last_seq_lstm = train_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)

lstm_predictions = []
current_seq_lstm = last_seq_lstm.copy()

for _ in range(len(val_df)):
    pred = lstm_model.predict(current_seq_lstm, verbose=0)[0][0]
    lstm_predictions.append(pred)
    current_seq_lstm = np.roll(current_seq_lstm, -1)
    current_seq_lstm[0, -1, 0] = pred

# Inverse transform
lstm_predictions = scaler.inverse_transform(np.array(lstm_predictions).reshape(-1, 1)).flatten()

# Calculate metrics
lstm_metrics = calculate_metrics(val_df['y'].values, lstm_predictions, 'LSTM')
print(f"\nLSTM Metrics:")
print(f"RMSE: {lstm_metrics['RMSE']:.4f}")
print(f"MAE: {lstm_metrics['MAE']:.4f}")
print(f"MAPE: {lstm_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot LSTM predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    lstm_predictions, 
    'LSTM',
    train_df['ds'].values,
    val_df['ds'].values
)

### 6.4 CNN-LSTM Hybrid

In [ ]:
# Build CNN-LSTM Model
cnn_lstm_model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(WINDOW_SIZE, 1)),
    MaxPooling1D(pool_size=2),
    LSTM(50, activation='relu'),
    Dense(1)
])

cnn_lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("CNN-LSTM Model Summary:")
cnn_lstm_model.summary()

In [ ]:
# Train CNN-LSTM
cnn_lstm_history = cnn_lstm_model.fit(
    X_train_lstm, y_train_lstm,  # Same data format as LSTM
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_lstm, y_val_lstm),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Make CNN-LSTM predictions
last_seq_cnn_lstm = train_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)

cnn_lstm_predictions = []
current_seq_cnn_lstm = last_seq_cnn_lstm.copy()

for _ in range(len(val_df)):
    pred = cnn_lstm_model.predict(current_seq_cnn_lstm, verbose=0)[0][0]
    cnn_lstm_predictions.append(pred)
    current_seq_cnn_lstm = np.roll(current_seq_cnn_lstm, -1)
    current_seq_cnn_lstm[0, -1, 0] = pred

# Inverse transform
cnn_lstm_predictions = scaler.inverse_transform(np.array(cnn_lstm_predictions).reshape(-1, 1)).flatten()

# Calculate metrics
cnn_lstm_metrics = calculate_metrics(val_df['y'].values, cnn_lstm_predictions, 'CNN-LSTM')
print(f"\nCNN-LSTM Metrics:")
print(f"RMSE: {cnn_lstm_metrics['RMSE']:.4f}")
print(f"MAE: {cnn_lstm_metrics['MAE']:.4f}")
print(f"MAPE: {cnn_lstm_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot CNN-LSTM predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    cnn_lstm_predictions, 
    'CNN-LSTM',
    train_df['ds'].values,
    val_df['ds'].values
)

## 7. Traditional Time Series Models

### 7.1 ARIMA (Autoregressive Integrated Moving Average)

In [ ]:
# Check stationarity
def adf_test(series):
    """Perform Augmented Dickey-Fuller test"""
    result = adfuller(series, autolag='AIC')
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value}')
    return result[1] < 0.05  # Returns True if stationary

print("ADF Test on Original Series:")
is_stationary = adf_test(train_df['y'].values)

In [ ]:
# Plot ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(train_df['y'], lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

plot_pacf(train_df['y'], lags=40, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.savefig('output/acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fit ARIMA model
# Using (p,d,q) = (7, 1, 7) based on weekly seasonality pattern
arima_order = (7, 1, 7)

print(f"Fitting ARIMA{arima_order}...")
arima_model = ARIMA(train_df['y'], order=arima_order)
arima_fitted = arima_model.fit()

print(arima_fitted.summary())

In [ ]:
# Make ARIMA predictions
arima_forecast = arima_fitted.forecast(steps=len(val_df))

# Calculate metrics
arima_metrics = calculate_metrics(val_df['y'].values, arima_forecast.values, 'ARIMA')
print(f"\nARIMA Metrics:")
print(f"RMSE: {arima_metrics['RMSE']:.4f}")
print(f"MAE: {arima_metrics['MAE']:.4f}")
print(f"MAPE: {arima_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot ARIMA predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    arima_forecast.values, 
    'ARIMA',
    train_df['ds'].values,
    val_df['ds'].values
)

### 7.2 SARIMA (Seasonal ARIMA)

In [ ]:
# Fit SARIMA model
# Seasonal period = 7 (weekly)
sarima_order = (1, 1, 1)
seasonal_order = (1, 1, 1, 7)  # (P, D, Q, S)

print(f"Fitting SARIMA{sarima_order}x{seasonal_order}...")
sarima_model = SARIMAX(train_df['y'], 
                       order=sarima_order, 
                       seasonal_order=seasonal_order,
                       enforce_stationarity=False,
                       enforce_invertibility=False)
sarima_fitted = sarima_model.fit(disp=False)

print(sarima_fitted.summary())

In [ ]:
# Make SARIMA predictions
sarima_forecast = sarima_fitted.forecast(steps=len(val_df))

# Calculate metrics
sarima_metrics = calculate_metrics(val_df['y'].values, sarima_forecast.values, 'SARIMA')
print(f"\nSARIMA Metrics:")
print(f"RMSE: {sarima_metrics['RMSE']:.4f}")
print(f"MAE: {sarima_metrics['MAE']:.4f}")
print(f"MAPE: {sarima_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot SARIMA predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    sarima_forecast.values, 
    'SARIMA',
    train_df['ds'].values,
    val_df['ds'].values
)

### 7.3 Prophet

In [ ]:
# Prepare data for Prophet
prophet_train = train_df.copy()
prophet_train.columns = ['ds', 'y']

# Create and fit Prophet model
print("Fitting Prophet model...")
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive'
)
prophet_model.fit(prophet_train)

In [ ]:
# Make Prophet predictions
prophet_future = prophet_model.make_future_dataframe(periods=len(val_df))
prophet_forecast = prophet_model.predict(prophet_future)

# Extract validation predictions
prophet_predictions = prophet_forecast['yhat'].values[-len(val_df):]

# Calculate metrics
prophet_metrics = calculate_metrics(val_df['y'].values, prophet_predictions, 'Prophet')
print(f"\nProphet Metrics:")
print(f"RMSE: {prophet_metrics['RMSE']:.4f}")
print(f"MAE: {prophet_metrics['MAE']:.4f}")
print(f"MAPE: {prophet_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot Prophet forecast
fig = prophet_model.plot(prophet_forecast, figsize=(14, 6))
plt.title('Prophet Forecast', fontsize=14)
plt.savefig('output/prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot Prophet components
fig = prophet_model.plot_components(prophet_forecast, figsize=(14, 10))
plt.savefig('output/prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot Prophet predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    prophet_predictions, 
    'Prophet',
    train_df['ds'].values,
    val_df['ds'].values
)

### 7.4 Exponential Smoothing (Holt-Winters)

In [ ]:
# Fit Holt-Winters model
print("Fitting Holt-Winters Exponential Smoothing...")

# Create time series index
train_ts = train_df.set_index('ds')['y']

hw_model = ExponentialSmoothing(
    train_ts,
    seasonal_periods=7,  # Weekly seasonality
    trend='add',
    seasonal='add',
    damped_trend=True
)
hw_fitted = hw_model.fit(optimized=True)

print("Holt-Winters model fitted successfully!")

In [ ]:
# Make Holt-Winters predictions
hw_forecast = hw_fitted.forecast(steps=len(val_df))

# Calculate metrics
hw_metrics = calculate_metrics(val_df['y'].values, hw_forecast.values, 'Holt-Winters')
print(f"\nHolt-Winters Metrics:")
print(f"RMSE: {hw_metrics['RMSE']:.4f}")
print(f"MAE: {hw_metrics['MAE']:.4f}")
print(f"MAPE: {hw_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot Holt-Winters predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    hw_forecast.values, 
    'Holt-Winters',
    train_df['ds'].values,
    val_df['ds'].values
)

### 7.5 XGBoost

In [ ]:
# Create features for XGBoost
def create_features(df, target_col='y'):
    """Create time series features from datetime index"""
    df = df.copy()
    df['dayofweek'] = df['ds'].dt.dayofweek
    df['quarter'] = df['ds'].dt.quarter
    df['month'] = df['ds'].dt.month
    df['year'] = df['ds'].dt.year
    df['dayofyear'] = df['ds'].dt.dayofyear
    df['dayofmonth'] = df['ds'].dt.day
    df['weekofyear'] = df['ds'].dt.isocalendar().week.astype(int)
    
    # Lag features
    for lag in [1, 7, 14, 30]:
        df[f'lag_{lag}'] = df[target_col].shift(lag)
    
    # Rolling features
    for window in [7, 14, 30]:
        df[f'rolling_mean_{window}'] = df[target_col].shift(1).rolling(window=window).mean()
        df[f'rolling_std_{window}'] = df[target_col].shift(1).rolling(window=window).std()
    
    return df

# Create features for training data
train_features = create_features(train_df)
train_features = train_features.dropna()

print(f"Training features shape: {train_features.shape}")
train_features.head()

In [ ]:
# Define feature columns
feature_cols = ['dayofweek', 'quarter', 'month', 'year', 'dayofyear', 
                'dayofmonth', 'weekofyear', 'lag_1', 'lag_7', 'lag_14', 'lag_30',
                'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14',
                'rolling_mean_30', 'rolling_std_30']

# Prepare training data
X_train_xgb = train_features[feature_cols]
y_train_xgb = train_features['y']

# Train XGBoost model
print("Training XGBoost model...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_xgb, y_train_xgb, eval_set=[(X_train_xgb, y_train_xgb)], 
              early_stopping_rounds=50, verbose=False)

print("XGBoost model trained successfully!")

In [ ]:
# Make XGBoost predictions for validation period
# We need to iteratively predict since we need lag features
xgb_predictions = []
current_data = train_df.copy()

for i in range(len(val_df)):
    # Create features for the next prediction
    next_date = val_df.iloc[i]['ds']
    next_row = pd.DataFrame({'ds': [next_date], 'y': [np.nan]})
    current_data = pd.concat([current_data, next_row], ignore_index=True)
    
    # Create features
    temp_features = create_features(current_data)
    temp_features = temp_features.iloc[-1:][feature_cols]
    
    # Predict
    pred = xgb_model.predict(temp_features)[0]
    xgb_predictions.append(pred)
    
    # Update current data with prediction
    current_data.iloc[-1, current_data.columns.get_loc('y')] = pred

# Calculate metrics
xgb_metrics = calculate_metrics(val_df['y'].values, np.array(xgb_predictions), 'XGBoost')
print(f"\nXGBoost Metrics:")
print(f"RMSE: {xgb_metrics['RMSE']:.4f}")
print(f"MAE: {xgb_metrics['MAE']:.4f}")
print(f"MAPE: {xgb_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot XGBoost predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    np.array(xgb_predictions), 
    'XGBoost',
    train_df['ds'].values,
    val_df['ds'].values
)

In [ ]:
# Plot feature importance
plt.figure(figsize=(10, 8))
xgb.plot_importance(xgb_model, max_num_features=15)
plt.title('XGBoost Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig('output/xgboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.6 Random Forest

In [ ]:
# Train Random Forest model
print("Training Random Forest model...")
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_xgb, y_train_xgb)

print("Random Forest model trained successfully!")

In [ ]:
# Make Random Forest predictions
rf_predictions = []
current_data_rf = train_df.copy()

for i in range(len(val_df)):
    next_date = val_df.iloc[i]['ds']
    next_row = pd.DataFrame({'ds': [next_date], 'y': [np.nan]})
    current_data_rf = pd.concat([current_data_rf, next_row], ignore_index=True)
    
    temp_features = create_features(current_data_rf)
    temp_features = temp_features.iloc[-1:][feature_cols]
    
    pred = rf_model.predict(temp_features)[0]
    rf_predictions.append(pred)
    
    current_data_rf.iloc[-1, current_data_rf.columns.get_loc('y')] = pred

# Calculate metrics
rf_metrics = calculate_metrics(val_df['y'].values, np.array(rf_predictions), 'Random Forest')
print(f"\nRandom Forest Metrics:")
print(f"RMSE: {rf_metrics['RMSE']:.4f}")
print(f"MAE: {rf_metrics['MAE']:.4f}")
print(f"MAPE: {rf_metrics['MAPE (%)']:.4f}%")

In [ ]:
# Plot Random Forest predictions
plot_predictions(
    train_df['y'].values, 
    val_df['y'].values, 
    np.array(rf_predictions), 
    'Random Forest',
    train_df['ds'].values,
    val_df['ds'].values
)

## 8. Model Comparison and Analysis

In [ ]:
# Collect all metrics
all_metrics = [
    mlp_metrics,
    cnn_metrics,
    lstm_metrics,
    cnn_lstm_metrics,
    arima_metrics,
    sarima_metrics,
    prophet_metrics,
    hw_metrics,
    xgb_metrics,
    rf_metrics
]

# Create comparison dataframe
comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df.sort_values('RMSE')

print("\n" + "="*60)
print("MODEL COMPARISON RESULTS (Sorted by RMSE)")
print("="*60)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# RMSE Comparison
colors = ['green' if x == comparison_df['RMSE'].min() else 'steelblue' for x in comparison_df['RMSE']]
axes[0].barh(comparison_df['Model'], comparison_df['RMSE'], color=colors)
axes[0].set_xlabel('RMSE')
axes[0].set_title('RMSE Comparison (Lower is Better)', fontsize=12)
axes[0].invert_yaxis()

# MAE Comparison
colors = ['green' if x == comparison_df['MAE'].min() else 'steelblue' for x in comparison_df['MAE']]
axes[1].barh(comparison_df['Model'], comparison_df['MAE'], color=colors)
axes[1].set_xlabel('MAE')
axes[1].set_title('MAE Comparison (Lower is Better)', fontsize=12)
axes[1].invert_yaxis()

# MAPE Comparison
colors = ['green' if x == comparison_df['MAPE (%)'].min() else 'steelblue' for x in comparison_df['MAPE (%)']]
axes[2].barh(comparison_df['Model'], comparison_df['MAPE (%)'], color=colors)
axes[2].set_xlabel('MAPE (%)')
axes[2].set_title('MAPE Comparison (Lower is Better)', fontsize=12)
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('output/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot all predictions together
plt.figure(figsize=(16, 8))

# Plot actual values
plt.plot(val_df['ds'], val_df['y'], label='Actual', color='black', linewidth=2)

# Plot all predictions
colors = plt.cm.tab10(np.linspace(0, 1, 10))
predictions_list = [
    ('MLP', mlp_predictions),
    ('CNN', cnn_predictions),
    ('LSTM', lstm_predictions),
    ('CNN-LSTM', cnn_lstm_predictions),
    ('ARIMA', arima_forecast.values),
    ('SARIMA', sarima_forecast.values),
    ('Prophet', prophet_predictions),
    ('Holt-Winters', hw_forecast.values),
    ('XGBoost', np.array(xgb_predictions)),
    ('Random Forest', np.array(rf_predictions))
]

for i, (name, pred) in enumerate(predictions_list):
    plt.plot(val_df['ds'], pred, label=name, color=colors[i], alpha=0.7, linestyle='--')

plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('All Models - Prediction Comparison on Validation Set', fontsize=14)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('output/all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save comparison results to CSV
comparison_df.to_csv('output/model_comparison_results.csv', index=False)
print("Results saved to output/model_comparison_results.csv")

## 9. Best Model Analysis and Recommendation

In [ ]:
# Identify best model
best_model = comparison_df.iloc[0]

print("\n" + "="*70)
print("BEST MODEL RECOMMENDATION")
print("="*70)
print(f"\nBest Model: {best_model['Model']}")
print(f"RMSE: {best_model['RMSE']:.4f}")
print(f"MAE: {best_model['MAE']:.4f}")
print(f"MAPE: {best_model['MAPE (%)']:.4f}%")

print("\n" + "-"*70)
print("TOP 3 MODELS:")
print("-"*70)
for i, row in comparison_df.head(3).iterrows():
    print(f"\n{row['Model']}:")
    print(f"  - RMSE: {row['RMSE']:.4f}")
    print(f"  - MAE: {row['MAE']:.4f}")
    print(f"  - MAPE: {row['MAPE (%)']:.4f}%")

In [ ]:
# Categorize models
deep_learning_models = ['MLP', 'CNN', 'LSTM', 'CNN-LSTM']
traditional_models = ['ARIMA', 'SARIMA', 'Prophet', 'Holt-Winters', 'XGBoost', 'Random Forest']

dl_avg_rmse = comparison_df[comparison_df['Model'].isin(deep_learning_models)]['RMSE'].mean()
trad_avg_rmse = comparison_df[comparison_df['Model'].isin(traditional_models)]['RMSE'].mean()

print("\n" + "="*70)
print("CATEGORY COMPARISON")
print("="*70)
print(f"\nDeep Learning Models Average RMSE: {dl_avg_rmse:.4f}")
print(f"Traditional Models Average RMSE: {trad_avg_rmse:.4f}")

if dl_avg_rmse < trad_avg_rmse:
    print("\n-> Deep Learning models perform better on average for this dataset.")
else:
    print("\n-> Traditional models perform better on average for this dataset.")

## 10. Conclusions and Key Findings

In [ ]:
print("""
============================================================================
                           KEY FINDINGS AND CONCLUSIONS
============================================================================

1. MODEL PERFORMANCE:
   - The best performing model is identified based on RMSE, MAE, and MAPE metrics.
   - Deep learning models (MLP, CNN, LSTM, CNN-LSTM) are effective at capturing
     complex temporal patterns but require more data and computational resources.
   - Traditional models (ARIMA, SARIMA, Prophet, Holt-Winters) are often more
     interpretable and can work well with smaller datasets.
   - Machine learning models (XGBoost, Random Forest) excel at feature engineering
     and can capture non-linear relationships effectively.

2. WHEN TO USE EACH MODEL TYPE:
   
   Deep Learning (MLP, CNN, LSTM, CNN-LSTM):
   - Large datasets with complex patterns
   - Long-term dependencies in data
   - When computational resources are available
   - When model interpretability is not critical
   
   Traditional Statistical (ARIMA, SARIMA, Holt-Winters):
   - Smaller datasets
   - Clear trend and seasonality patterns
   - When interpretability is important
   - Quick baseline models
   
   Prophet:
   - Business time series with multiple seasonality
   - When automatic seasonality detection is needed
   - When handling holidays and special events
   
   Machine Learning (XGBoost, Random Forest):
   - When rich feature engineering is possible
   - When external variables are available
   - For capturing non-linear relationships
   - When feature importance analysis is needed

3. RECOMMENDATION:
   - Start with simple models (ARIMA, Holt-Winters) as baselines
   - Use Prophet for quick and robust forecasts
   - Consider XGBoost/Random Forest when rich features are available
   - Use deep learning (LSTM, CNN) for complex patterns and large datasets
   - Always validate models on out-of-sample data

4. FUTURE IMPROVEMENTS:
   - Ensemble methods combining multiple models
   - Hyperparameter tuning for each model
   - Cross-validation for more robust evaluation
   - Adding external regressors and features
   - Handling multiple time series (hierarchical forecasting)

============================================================================
""")

In [ ]:
# Final summary table
print("\n" + "="*70)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*70)
print(comparison_df.to_string(index=False))
print("\nBest Model: ", comparison_df.iloc[0]['Model'])